In [6]:
# ============================================================
# IMPORT THE REQUIRED LIBRARIES
# ============================================================

import torch
import torch.nn as nn

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer


# ============================================================
# STEP 1: PREPARE THE TRAINING DATA
# ============================================================

# These are the example movie reviews that the model will learn from.
reviews = [
    "the movie was great",
    "i hated this film",
    "it was absolutely amazing",
    "what a waste of time"
]

# Each label describes the sentiment of the matching review.
#
# 1 means positive
# 0 means negative
labels = [1, 0, 1, 0]


# ============================================================
# STEP 2: CREATE AND TRAIN THE TOKENIZER
# ============================================================

# A tokenizer converts text into numbers that the neural
# network can understand.
tokenizer = Tokenizer(
    BPE(unk_token="[UNK]")
)

# Split each sentence at whitespace before creating tokens.
tokenizer.pre_tokenizer = Whitespace()

# Configure the tokenizer trainer.
#
# [PAD] is used to make all sentences the same length.
# [UNK] represents words or characters the tokenizer does not know.
trainer = BpeTrainer(
    vocab_size=20,
    special_tokens=["[PAD]", "[UNK]"]
)

# Train the tokenizer using our movie reviews.
tokenizer.train_from_iterator(
    reviews,
    trainer=trainer
)

# Find the numerical ID of the padding token.
pad_id = tokenizer.token_to_id("[PAD]")

# Find the actual size of the tokenizer's vocabulary.
vocab_size = tokenizer.get_vocab_size()

print("Padding token ID:", pad_id)
print("Vocabulary size:", vocab_size)


# ============================================================
# STEP 3: CONVERT THE REVIEWS INTO FIXED-LENGTH NUMBER LISTS
# ============================================================

# Convert each review into a list of token IDs.
raw_ids = [
    tokenizer.encode(review).ids
    for review in reviews
]

print("\nRaw token IDs:")
print(raw_ids)

# Every review will contain exactly five token IDs.
max_len = 5

padded_ids = []

for token_ids in raw_ids:
    # Keep only the first five token IDs.
    truncated_ids = token_ids[:max_len]

    # Calculate how many padding tokens are needed.
    pad_amount = max_len - len(truncated_ids)

    # Add padding tokens to the end of the sequence.
    padded_sequence = (
        truncated_ids + [pad_id] * pad_amount
    )

    padded_ids.append(padded_sequence)

print("\nPadded token IDs:")
print(padded_ids)


# ============================================================
# STEP 4: CONVERT THE DATA INTO PYTORCH TENSORS
# ============================================================

# X contains the token IDs for the reviews.
#
# Token IDs must use the torch.long data type because they
# will be passed into an embedding layer.
X = torch.tensor(
    padded_ids,
    dtype=torch.long
)

# Y contains the correct sentiment labels.
#
# BCEWithLogitsLoss expects floating-point labels with the
# same shape as the model's output.
Y = torch.tensor(
    labels,
    dtype=torch.float32
).reshape(-1, 1)

print("\n========== INPUT DATA X ==========")
print(X)
print("Number of reviews:", len(X))
print("Shape of X:", X.shape)

print("\n========== LABELS Y ==========")
print(Y)
print("Shape of Y:", Y.shape)


# ============================================================
# STEP 5: CREATE THE SENTIMENT ANALYSIS MODEL
# ============================================================

class SentimentNet(nn.Module):
    """
    A small neural network for classifying a sentence as
    positive or negative.
    """

    def __init__(self, vocab_size, embed_dim, pad_id):
        super().__init__()

        # Save the padding token ID so we can ignore padding
        # while calculating the sentence representation.
        self.pad_id = pad_id

        # The embedding layer converts each token ID into a
        # vector of learned numbers.
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id
        )

        # The linear layer converts the sentence vector into
        # one output value called a logit.
        self.linear = nn.Linear(
            in_features=embed_dim,
            out_features=1
        )

    def forward(self, x):
        # Convert token IDs into embedding vectors.
        #
        # Input shape:
        # [batch_size, sentence_length]
        #
        # Output shape:
        # [batch_size, sentence_length, embedding_dimension]
        embedded = self.embedding(x)

        # Create a mask:
        #
        # Real token = 1
        # Padding token = 0
        mask = (
            x != self.pad_id
        ).unsqueeze(-1).float()

        # Set the padding embeddings to zero.
        masked_embeddings = embedded * mask

        # Count the real tokens in each sentence.
        token_counts = mask.sum(dim=1)

        # Prevent division by zero if a sentence contains
        # only padding tokens.
        token_counts = token_counts.clamp(min=1.0)

        # Average the embeddings of the real tokens.
        sentence_meaning = (
            masked_embeddings.sum(dim=1) / token_counts
        )

        # Produce one raw prediction score for each sentence.
        logits = self.linear(sentence_meaning)

        return logits


# Create the neural network.
model = SentimentNet(
    vocab_size=vocab_size,
    embed_dim=8,
    pad_id=pad_id
)

print("\n========== MODEL ==========")
print(model)


# ============================================================
# STEP 6: PREPARE THE LOSS FUNCTION AND OPTIMIZER
# ============================================================

# BCEWithLogitsLoss measures how far the model's predictions
# are from the correct binary labels.
#
# It automatically applies the sigmoid calculation internally.
loss_fn = nn.BCEWithLogitsLoss()

# Adam updates the model's weights and embeddings during training.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.05
)


# ============================================================
# STEP 7: TRAIN THE MODEL
# ============================================================

print("\n========== TRAINING STARTED ==========")

# Put the model into training mode.
model.train()

number_of_epochs = 200

for epoch in range(number_of_epochs):
    # Remove gradients left over from the previous training step.
    optimizer.zero_grad()

    # Ask the model to make predictions.
    logits = model(X)

    # Compare the predictions with the correct labels.
    loss = loss_fn(logits, Y)

    # Calculate how each parameter contributed to the error.
    loss.backward()

    # Update the model's parameters.
    optimizer.step()

    # Show the progress after every 20 epochs.
    if (epoch + 1) % 20 == 0:
        # Convert the raw logits into probabilities from 0 to 1.
        probabilities = torch.sigmoid(logits)

        # A score of 0.5 or higher means positive.
        predicted_labels = (
            probabilities >= 0.5
        ).float()

        # Calculate the model's accuracy on the training data.
        accuracy = (
            predicted_labels == Y
        ).float().mean().item()

        print(
            f"Epoch {epoch + 1:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Accuracy: {accuracy:.2%}"
        )

print("\nTraining complete!")


# ============================================================
#

Padding token ID: 0
Vocabulary size: 22

Raw token IDs:
[[16, 8, 5, 11, 13, 18, 9, 5, 19, 2, 15, 7, 14, 5, 2, 16], [9, 8, 2, 16, 5, 4, 16, 8, 9, 15, 6, 9, 10, 11], [9, 16, 19, 2, 15, 2, 3, 15, 13, 10, 17, 16, 5, 10, 20, 2, 11, 2, 21, 9, 12, 7], [19, 8, 2, 16, 2, 19, 2, 15, 16, 5, 13, 6, 16, 9, 11, 5]]

Padded token IDs:
[[16, 8, 5, 11, 13], [9, 8, 2, 16, 5], [9, 16, 19, 2, 15], [19, 8, 2, 16, 2]]

========== INPUT DATA X ==========
tensor([[16,  8,  5, 11, 13],
        [ 9,  8,  2, 16,  5],
        [ 9, 16, 19,  2, 15],
        [19,  8,  2, 16,  2]])
Number of reviews: 4
Shape of X: torch.Size([4, 5])

========== LABELS Y ==========
tensor([[1.],
        [0.],
        [1.],
        [0.]])
Shape of Y: torch.Size([4, 1])

========== MODEL ==========
SentimentNet(
  (embedding): Embedding(22, 8, padding_idx=0)
  (linear): Linear(in_features=8, out_features=1, bias=True)
)

========== TRAINING STARTED ==========
Epoch  20 | Loss: 0.1009 | Accuracy: 100.00%
Epoch  40 | Loss: 0.0039 | Accura